In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv("../data/diabetes_data/diabetic_data.csv")
df = df.replace('?', np.nan)
df = df.drop(columns=['weight', 'payer_code', 'medical_specialty'])
df['race'] = df['race'].fillna('Unknown')

df = df.sort_values('encounter_id')
df = df.drop_duplicates(subset='patient_nbr', keep='first')

exclude_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(exclude_ids)]

df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

categorical_features = [
    'race', 'gender', 'age', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id',
    'A1Cresult', 'max_glu_serum', 'change', 'diabetesMed', 'insulin'
]

X = df[numeric_features + categorical_features]
y = df['readmitted_binary']
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((55978, 76), (13995, 76))

Random Forest

In [2]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

              precision    recall  f1-score   support

           0       0.94      0.67      0.78     12740
           1       0.14      0.53      0.22      1255

    accuracy                           0.66     13995
   macro avg       0.54      0.60      0.50     13995
weighted avg       0.86      0.66      0.73     13995

ROC-AUC: 0.6392482190547074


Gradient Boosting

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb_model = HistGradientBoostingClassifier(max_depth=6, class_weight='balanced', random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_gb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_gb))

              precision    recall  f1-score   support

           0       0.93      0.69      0.80     12740
           1       0.14      0.51      0.22      1255

    accuracy                           0.68     13995
   macro avg       0.54      0.60      0.51     13995
weighted avg       0.86      0.68      0.74     13995

ROC-AUC: 0.6450413416975739


Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_rf = cross_val_score(rf_model, X, y, cv=cv, scoring='roc_auc')
print("Random forest CV ROC-AUC:", scores_rf, "→ mean:", scores_rf.mean())

scores_gb = cross_val_score(gb_model, X, y, cv=cv, scoring='roc_auc')
print("Gradient boosting CV ROC-AUC:", scores_gb, "→ mean:", scores_gb.mean())

Random forest CV ROC-AUC: [0.64996016 0.63004435 0.64609623 0.63684479 0.6461787 ] → snitt: 0.6418248449983462
Gradient boosting CV ROC-AUC: [0.65108217 0.63301416 0.65026015 0.64452863 0.64223101] → snitt: 0.6442232234169569


Model comparison: Logistic regression, random forest, and gradient boosting all landed within ~0.005 ROC-AUC of each other (~0.64), suggesting this feature set has a hard ceiling on predictive power that model complexity alone can't push past. Gradient boosting selected as the final model (CV ROC-AUC 0.644) for its slight edge and compatibility with SHAP.